In [ ]:
# Notebook version: 1.0.0 (DSPCL; see README for change history)
from __future__ import print_function
from __future__ import absolute_import
from __future__ import unicode_literals

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.patches as patches
import math as m
from time import time

from scipy import special
import pandas as pd

from IPython.display import HTML, display
import ipywidgets as widgets

%matplotlib inline

deg2rad = lambda deg: deg / 180. * np.pi
rad2deg = lambda rad: rad * 180. / np.pi

import warnings
warnings.filterwarnings('ignore')

# --- module-specific imports go here (add a commented pip/conda line above each; standard stack only) ---


# --- DSP_COLORS palette (DSPCL S.2.2): POTE1_COLORS unchanged + DSP extensions ---
DSP_COLORS = {
    # ===== Inherited from POTE1_COLORS (STYLE_GUIDE_V3_R3 S.2.2) - canonical, do not drift =====
    # --- Solar system / space (Module 2) ---
    'sun':              '#FFD700',   # Gold
    'terrestrial':      '#c1440e',   # Mars red
    'giant':            '#c88b3a',   # Jupiter tan
    'space_bg':         '#0d1b2a',   # Deep space dark blue
    # --- Earth structural layers (Modules 3, 4, 5) ---
    'inner_core':       '#b5121b',   # Deep red
    'outer_core':       '#e8401c',   # Orange-red
    'lower_mantle':     '#e07b39',   # Burnt orange
    'upper_mantle':     '#e8a838',   # Amber
    'asthenosphere':    '#c8b870',   # Warm tan (partial melt)
    'oceanic_crust':    '#2e6fa3',   # Ocean blue
    'continental_crust':'#a08050',   # Sandy brown
    'water':            '#aad4f5',   # Pale blue
    # --- Plate tectonics (Module 3) ---
    'divergent':        '#d62728',   # Red
    'convergent':       '#1f77b4',   # Blue
    'transform':        '#2ca02c',   # Green
    'hotspot':          '#ff7f0e',   # Orange
    'slab':             '#17becf',   # Cyan (cold slab)
    # --- Geophysical fields (Modules 4–10) ---
    'gravity_pos':      '#d62728',   # Positive anomaly red
    'gravity_neg':      '#1f77b4',   # Negative anomaly blue
    'mag_normal':       '#2c2c2c',   # Normal polarity black
    'mag_reversed':     '#f0f0f0',   # Reversed polarity white/light
    'heat_high':        '#d62728',   # High heat flow red
    'heat_low':         '#1f77b4',   # Low heat flow blue
    # --- General purpose ---
    'highlight':        '#ff7f0e',   # Orange accent
    'neutral':          '#7f7f7f',   # Medium gray
    'earth':            '#4fa8d5',   # Earth blue (general Earth reference)
    'accent':           '#ffffff',   # White accent
    # ===== DSP extensions =====
    # --- Domains ---
    'time_domain':      '#1f77b4',   # Blue: signals plotted against time/space
    'freq_domain':      '#d62728',   # Red: spectra plotted against frequency/wavenumber
    'phase':            '#9467bd',   # Purple: phase spectra
    # --- Continuous vs sampled ---
    'continuous':       '#7f7f7f',   # Gray: underlying analog signal
    'sampled':          '#1f77b4',   # Blue: discrete samples (stems/markers)
    'alias':            '#ff7f0e',   # Orange: aliased components / spectral replicas
    # --- Systems ---
    'input':            '#1f77b4',   # Blue: system input x
    'impulse_resp':     '#2ca02c',   # Green: impulse response h
    'output':           '#d62728',   # Red: system output y
    'noise':            '#bcbd22',   # Olive: additive noise
    # --- Windows, filters, Z-plane ---
    'window':           '#8c564b',   # Brown: window/taper functions
    'passband':         '#aad4f5',   # Pale blue: filter passband shading
    'pole':             '#d62728',   # Red: poles (x markers)
    'zero':             '#1f77b4',   # Blue: zeros (o markers)
    'unit_circle':      '#2c2c2c',   # Near-black: |z| = 1
}

# --- rcParams defaults (S.2.1) ---
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'sans-serif',
    'font.size':        12,
    'axes.titlesize':   14,
    'axes.labelsize':   12,
    'xtick.labelsize':  11,
    'ytick.labelsize':  11,
    'legend.fontsize':  11,
    'figure.dpi':       120,
    'lines.linewidth':  2.0,
    'axes.grid':        True,
    'grid.alpha':       0.3,
})

# --- Animation embed size limit (MB) — increase if animations are large ---
plt.rcParams['animation.embed_limit'] = 50

# --- Full-width notebook display ---
display(HTML("<style>.container { width:100% !important; }</style>"))

# --- Robust figure/table centering (classic Notebook 6 + JupyterLab) ---
display(HTML("""
<style>
.output_png { text-align: center; }
.output_area .output_png img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output img { display: block; margin-left: auto; margin-right: auto; }
.jp-OutputArea-output table { margin-left: auto; margin-right: auto; }
.output_area table { margin-left: auto; margin-right: auto; }
/* center every table: Markdown tables and pandas/HTML tables in outputs (DSPCL ruling 2026-10-05) */
.jp-RenderedHTMLCommon table, .rendered_html table, .jp-RenderedMarkdown table {
    margin-left: auto !important; margin-right: auto !important; }
</style>
"""))

# --- Code-cell toggle (hidden by default for cleaner reading) ---
display(HTML('''<style>
.code-toggle-btn {
    background: #2c3e50; color: white; border: none;
    padding: 5px 12px; border-radius: 4px; cursor: pointer;
    font-size: 13px; margin: 4px 0;
}
.code-toggle-btn:hover { background: #34495e; }
</style>
<script>
var pote1_code_shown = true;
function pote1ToggleCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    var nodes = document.querySelectorAll(sel);
    pote1_code_shown = !pote1_code_shown;
    nodes.forEach(function(el){ el.style.display = pote1_code_shown ? '' : 'none'; });
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = pote1_code_shown ? 'Hide code' : 'Show code';
}
function pote1HideCode() {
    var sel = ['.code_cell .input_area', '.jp-CodeCell .jp-InputArea-editor',
               '.jp-CodeCell .jp-InputPrompt', '.code_cell .prompt.input_prompt'].join(',');
    document.querySelectorAll(sel).forEach(function(el){ el.style.display='none'; });
    pote1_code_shown = false;
    var lbl = document.getElementById('pote1-toggle-label');
    if (lbl) lbl.textContent = 'Show code';
}
/* Hide on load; retry briefly while the notebook DOM finishes rendering */
setTimeout(pote1HideCode, 300);
setTimeout(pote1HideCode, 1200);
</script>
<button class="code-toggle-btn" onclick="pote1ToggleCode()">
  <span id="pote1-toggle-label">Show code</span>
</button>
<span style="font-size:12px; color:#555; margin-left:8px;">
  Only the Python code is hidden — text and figures stay visible.
</span>'''))

In [ ]:
# Course title banner (Python-generated): a seismic gather, a sampled waveform, and the gather's computed spectrum
import matplotlib.patheffects as pe
from matplotlib.colors import LinearSegmentedColormap

_C = DSP_COLORS

def _ricker(t, f0):
    a = (np.pi * f0 * t) ** 2
    return (1 - 2 * a) * np.exp(-a)

def glow_text(ax, x, y, txt, fs):
    ax.text(x, y, txt, ha='center', va='center', fontsize=fs, color='white', fontweight='bold', zorder=20,
            path_effects=[pe.withStroke(linewidth=3.5, foreground='#0a1830'), pe.Normal()])

# --- synthetic shot gather: three reflections with hyperbolic moveout, Ricker source wavelet ---
_dt, _nt, _ntr, _f0 = 0.002, 400, 12, 30.0          # 2 ms sampling, 0.8 s, 12 traces, 30 Hz Ricker
_t = np.arange(_nt) * _dt
_off = np.linspace(0, 800, _ntr)                     # offsets (m)
_gather = np.zeros((_ntr, _nt))
for _t0, _v, _amp in [(0.14, 1900, 1.0), (0.34, 2300, -0.75), (0.56, 2700, 0.6)]:
    _tx = np.sqrt(_t0 ** 2 + (_off / _v) ** 2)
    _gather += _amp * _ricker(_t[None, :] - _tx[:, None], _f0)

# --- amplitude spectrum of the gather (mean over traces), normalized ---
_freq = np.fft.rfftfreq(_nt, _dt)
_spec = np.abs(np.fft.rfft(_gather, axis=1)).mean(axis=0)
_spec /= _spec.max()

fig, ax = plt.subplots(figsize=(14, 4.4))
ax.set_xlim(0, 14); ax.set_ylim(0, 4.4); ax.axis('off'); ax.grid(False)
ax.imshow(np.linspace(0, 1, 256).reshape(-1, 1), extent=[0, 14, 0, 4.4], aspect='auto', origin='lower', zorder=0,
          cmap=LinearSegmentedColormap.from_list('space', ['#04070f', '#0b1622', '#122438', '#0b1622']))
_rng = np.random.default_rng(7)
ax.scatter(_rng.uniform(0, 14, 220), _rng.uniform(0, 4.4, 220), s=_rng.uniform(0.3, 5, 220),
           c='white', alpha=0.45, zorder=1, edgecolors='none')

# --- left: wiggle-trace gather with variable-area fill (time increases downward) ---
_x0, _x1, _ytop, _ybot = 0.35, 3.15, 4.05, 0.35
_xs = np.linspace(_x0 + 0.12, _x1 - 0.12, _ntr)
_dx = (_xs[1] - _xs[0]) * 1.4
_y = _ytop - (_t / _t[-1]) * (_ytop - _ybot)
for _xc, _tr in zip(_xs, _gather):
    _w = _xc + _dx * _tr
    ax.plot(_w, _y, color='#cfe6ff', lw=0.8, zorder=4)
    ax.fill_betweenx(_y, _xc, _w, where=_tr > 0, color=_C['earth'], alpha=0.9, lw=0, zorder=3)

# --- center: continuous chirp with its samples (continuous vs sampled) ---
_xc0, _xc1, _yc = 4.0, 10.0, 2.15
_s = np.linspace(0, 1, 800)
_chirp = lambda s: 0.32 * np.sin(2 * np.pi * (2.0 * s + 5.0 * s ** 2)) * np.sin(np.pi * s) ** 0.5   # 2 -> 12 cycles, tapered ends
ax.plot(_xc0 + (_xc1 - _xc0) * _s, _yc + _chirp(_s), color=_C['continuous'], lw=1.2, alpha=0.9, zorder=4)
_sk = np.linspace(0, 1, 61)
_xk = _xc0 + (_xc1 - _xc0) * _sk
ax.vlines(_xk, _yc, _yc + _chirp(_sk), color=_C['highlight'], lw=1.0, alpha=0.9, zorder=5)
ax.scatter(_xk, _yc + _chirp(_sk), s=9, color=_C['highlight'], zorder=6, edgecolors='none')

# --- right: amplitude spectrum of the gather on the left ---
_sx0, _sx1, _sy0, _sy1 = 10.65, 13.55, 0.45, 3.85
_fmax = 100.0
_m = _freq <= _fmax
_fx = _sx0 + (_sx1 - _sx0) * _freq[_m] / _fmax
_fy = _sy0 + (_sy1 - _sy0) * _spec[_m]
ax.fill_between(_fx, _sy0, _fy, color=_C['freq_domain'], alpha=0.55, lw=0, zorder=3)
ax.plot(_fx, _fy, color='#ffb3a8', lw=1.6, zorder=4)
ax.plot([_sx0, _sx1], [_sy0, _sy0], color='#cfe6ff', lw=1.0, alpha=0.7, zorder=4)
ax.text(_sx1, _sy0 - 0.17, 'f', color='#cfe6ff', fontsize=12, ha='right', va='center', style='italic', zorder=5)
ax.text(_x0 + 0.02, _ybot - 0.17, 't ↓', color='#cfe6ff', fontsize=12, ha='left', va='center', style='italic', zorder=5)

glow_text(ax, 7.0, 3.78, "GPGN 404", 30)
glow_text(ax, 7.0, 3.08, "Digital Signal Processing", 27)
glow_text(ax, 7.0, 1.22, "Professor Jeffrey Shragge", 22)
glow_text(ax, 7.0, 0.58, "Geophysics Dept, Colorado School of Mines", 20)

fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
plt.show()

<center><h1>Module 3 &mdash; Complex Numbers</h1></center>

Almost every tool in this course (Fourier series and transforms, the DFT, filters and the Z-transform) is written with complex numbers,
because a single complex exponential $e^{i\omega t}$ carries both the amplitude and the phase of an oscillation. A seismic trace is real, but the
fastest way to shift, filter or combine its oscillations is to treat each one as a rotating arrow in the complex plane. This module refreshes
complex arithmetic, the complex plane and polar form, then builds Euler's formula and the geometry of multiplication and roots, and finally
complex functions and analyticity, which we need for transforms and filters. Most of it will be familiar from MATH 213 and MATH 348; the
new part is reading each result as something that happens to a signal.

# Outline
- [3.1 Why the Complex Plane?](#3.1-Why-the-Complex-Plane?)
  - [3.1.1 Forward and inverse integral transforms](#3.1.1-Forward-and-inverse-integral-transforms)
  - [3.1.2 The Fourier transform pair](#3.1.2-The-Fourier-transform-pair)
  - [3.1.3 Three questions about the transform](#3.1.3-Three-questions-about-the-transform)
- [3.2 Complex Arithmetic](#3.2-Complex-Arithmetic)
- [3.3 The Complex Plane](#3.3-The-Complex-Plane)
  - [3.3.1 Points and vectors](#3.3.1-Points-and-vectors)
  - [3.3.2 Polar representation](#3.3.2-Polar-representation)
  - [3.3.3 Regions in the complex plane](#3.3.3-Regions-in-the-complex-plane)
- [3.4 The Complex Exponential](#3.4-The-Complex-Exponential)
- [3.5 Geometry of Multiplication and Division](#3.5-Geometry-of-Multiplication-and-Division)
- [3.6 Roots of a Complex Number](#3.6-Roots-of-a-Complex-Number)
- [3.7 Complex Functions](#3.7-Complex-Functions)
  - [3.7.1 Limits and continuity](#3.7.1-Limits-and-continuity)
  - [3.7.2 The derivative of a complex function](#3.7.2-The-derivative-of-a-complex-function)
  - [3.7.3 Analytic functions](#3.7.3-Analytic-functions)
  - [3.7.4 The Cauchy–Riemann equations](#3.7.4-The-Cauchy–Riemann-equations)
- [Microquestions](#Microquestions)

<a name="3.1"></a>
## 3.1 Why the Complex Plane?

In many instances in geophysical digital signal processing we apply [integral transforms](https://en.wikipedia.org/wiki/Integral_transform) that project our geophysical data from one domain to another.  The reason why we do this is that many of the problems we are trying to solve (e.g., filtering out unwanted components, signal-noise separation, etc) are easier to solve in the transform domain. Ideally, the integral transform is **invertible** in the sense that it has an **inverse transform** that allows you to bring your (filtered, separated, etc) data back to the original domain without losing any of its information content.

<a name="3.1.1"></a>
### 3.1.1 Forward and inverse integral transforms

Mathematically, we may write the **integral transform** as the following:


$$(Tg)(u) = \int_{t_1}^{t_2} g(t)\, K(t,u) \,\mathrm{d}t \tag{1}$$

where the input is a function $g(t)$ of variable $t$, and the output is another function $Tg$ of transform variable $u$. Often we symbolize the output of this transform as $\widehat{G}(u)$.  There are numerous useful integral transforms, where each is specified by a choice of the [integration kernel function](https://en.wikipedia.org/wiki/Integral_transform) $K(u,t)$ of two variables - $t$ and $u$ - as well as the limits applied - $t_1$ and $t_2$.

Some kernels have an associated inverse kernel $K^{-1}(t,u)$, which roughly speaking, delivers an **inverse transform**:

$$ g(t) =  \int_{u_1}^{u_2}  (Tg)(u) \, K^{-1}(t,u) \,\mathrm{d}u \tag{2} $$

when applied with the appropriate limits $u_1$ and $u_2$. There are many [different integral transforms](https://en.wikipedia.org/wiki/Integral_transform#Table_of_transforms) that are used across a whole range of the physical sciences and mathematics.

<a name="3.1.2"></a>
### 3.1.2 The Fourier transform pair

The most important set of integral transforms that you will likely experience over your career is the Forward and Inverse Fourier Transform pair (denoted $\mathcal{F}$ and $\mathcal{F}^{-1}$).  By defining $u=f$, we see that the kernel functions for these transforms, $K_{\mathcal{F}}(f,t)$ and $K^{-1}_{\mathcal{F}}(f,t)$, are given by:

$$K_{\mathcal{F}}(f,t) = e^{-i2\pi ft} \quad \mathrm{and} \quad K^{-1}_{\mathcal{F}}(f,t)=e^{i 2\pi f t} \tag{3}$$

In this course, we will sometimes use the **angular frequency** definition $\omega = 2\pi f$ so that we may write 

$$K_{\mathcal{F}}(\omega,t) = e^{-i \omega t} \quad \mathrm{and} \quad K^{-1}_{\mathcal{F}}(\omega,t)=e^{i\omega t} \tag{4}$$

When we include the limits $t_1=u_1=-\infty$ and $t_2=u_2=\infty$ we may write:

$$\widehat{G}(\omega)\equiv (\mathcal{F}g)(\omega) = \int_{-\infty}^{\infty} g(t) \, e^{-i \omega t} \, \mathrm{d}t \quad \mathrm{and} \quad g(t) = \frac{1}{2\pi} \int_{-\infty}^{\infty}  (\mathcal{F}g)(\omega) \, e^{i\omega t} \,\mathrm{d}\omega = \frac{1}{2\pi} \int_{-\infty}^{\infty} \widehat{G}(\omega) \, e^{i\omega t} \,\mathrm{d}\omega \tag{5}$$

The factor $1/(2\pi)$ on the inverse transform appears because $\mathrm{d}\omega = 2\pi\,\mathrm{d}f$; with the frequency $f$ of equation (3)
no factor is needed. This is the convention used throughout the course (Module 5).

<div class="alert alert-info">
<b>Sign convention:</b> This course uses $e^{-i\omega t}$ as the <i>forward</i> kernel and $e^{+i\omega t}$ for the inverse (equation 4).
Some physics texts, and many PHGN 200 treatments of AC phasors, use the opposite sign. Nothing physical changes, but the sign of every
phase in a spectrum flips, so always check which convention a formula assumes.
</div>

<a name="3.1.3"></a>
### 3.1.3 Three questions about the transform

The above Fourier Transform process looks pretty straightforward to compute.  However, there are a number of fundamental questions that need to be addressed:

(1) Does the formulation work for all $g(t)$?
   * **NO!** There are some restrictions on the types of $g(t)$ to which we can apply the Fourier Transform. For example, does it have "finite" (good!) or "infinite" (bad!) energy in the signal?
    
    
(2) Is it possible to specify some $g(t)$ for which the Fourier Transform "fails"?
   * **YES!** It is actually very easy to specify a function that would "fail" - e.g., $g(t) = e^{-t} (1-t)^{-1}$ around $t=1$.  This is quite common when we are using designer functions (i.e., "filters") to do specific filtering tasks. 
    
    
(3) Are we able to analyze these types of signals?
   * **YES!** Just because a signal's transform is not well defined for real frequencies $\omega$ does not mean we have to give up: we can still study it by letting the transform variable become complex.  Doing so leads to some course material that requires us to be comfortable with complex numbers and some of the fundamental principles of complex analysis.  Hopefully, you have encountered these topics in your previous courses and just need a short review.

<a name="3.2"></a>
## 3.2 Complex Arithmetic

You have probably encountered the expression $i=\sqrt{-1}$ with the imaginary unit many times before - likely first with certain solutions to the quadratic equation:

$$ax^2+bx+c=0 \tag{6}$$

given by

$$x=\frac{-b \pm \sqrt{b^2-4ac}}{2a} \tag{7}$$

When $b^2 < 4ac$, no real number $x$ satisfies equation (6).  The solutions in this case are termed **complex numbers**.  We have also assumed the existence of certain arithmetic operations on complex numbers that largely parallel our experience with real numbers (generally due to some sort of *mathematical symmetry*). However, we do not need to delve into such a detailed discussion for this course.

We first consider simple powers of the imaginary unit $i=\sqrt{-1}$:

* $i^0 = 1$
* $i^1 = i$
* $i^2 = \sqrt{-1}\sqrt{-1} = -1$
* $i^3 = i^2i=-i$
* $i^4 = i^2 i^2=(-1)(-1)=1$
* $i^5 = i$

<div class="alert alert-info" role="alert">
<b>Note:</b> Multiplying by $i$ rotates a complex number by $+90^\circ$ without changing its length, so each power in the list is the previous one turned a quarter turn. In general, multiplying adds angles and taking a root divides them (Sections 3.5 and 3.6).
</div>

When performing the operations of addition, subtraction, multiplication and division we treat $i$ as though it were an algebraic constant.  For example, if $z_1=3+2i$ and $z_2=2-i$ then:

$$\begin{eqnarray}
{\rm Addition:}\quad z_1+z_2 = (3+2i)+(2-i) &=& 5+i \tag{8}\\
{\rm Subtraction:}\quad  z_1-z_2=(3+2i)-(2-i) &=& 1+3i \tag{9}\\
{\rm Multiplication:}\quad  z_1z_2=(3+2i)(2-i)  &=& 6-3i+4i-2i^2 = 6+i+2 = 8+i\tag{10}\\ 
\end{eqnarray}$$

Division requires remembering a simple trick.  To compute $\frac{z_1}{z_2}=\frac{3+2i}{2-i}$ we first multiply both the numerator and denominator by the **complex conjugate** (i.e., if $z=a+ib$ then $\overline{z}=\overline{a+ib} = a-ib$ where $a, b \in \mathbb{R}$ and the overbar indicates a complex conjugate) of the term in the denominator.  Thus,

$${\rm Division:}\quad \frac{z_1}{z_2} = \frac{3+2i}{2-i} = \frac{(3+2i)(2+i)}{(2-i)(2+i)} = \frac{4+7i}{5} \tag{11}$$ 

This process is also called **rationalizing the denominator**.

We often want to extract out just the real or imaginary components of a complex number, which is written notationally using the **real and imaginary operators**, $\Re$ and $\Im$, respectively. Thus, if $z=a+ib$ we have the following:

$$\Re(z)=a \tag{12}$$ 
and
$$\Im(z)=b \tag{13}$$

Note that the result of either operation is a **real number**.

<b>Quick Problems</b>
(1) Let $z=3-3i$ and $w=1+\sqrt{3}i$.  Find the result of the following operations:

   (a) $z+w$
   
   (b) $z-w$
   
   (c) $z\,w$
   
   (d) $z\,/\,w$

Let's test our answers using the **sympy** package (symbolic python):

In [ ]:
# Check Quick Problem (1): exact arithmetic with sympy if it is installed, otherwise Python complex numbers
try:
    import sympy as sm
    z = 3 - 3*sm.I
    w = 1 + sm.sqrt(3)*sm.I
    print('Q1a: z+w =', sm.expand(z + w))
    print('Q1b: z-w =', sm.expand(z - w))
    print('Q1c: z*w =', sm.expand(z*w))
    print('Q1d: z/w =', sm.simplify(sm.expand(sm.radsimp(z/w))))
except ImportError:
    print('(sympy not installed: showing decimal values; install it with  conda install sympy  for exact forms)')
    z, w = 3 - 3j, 1 + np.sqrt(3)*1j
    for name, val in (('z+w', z + w), ('z-w', z - w), ('z*w', z*w), ('z/w', z/w)):
        print(f'{name} = {val.real:.4f} {val.imag:+.4f}i')

(2) Let $z=a+ib$ and $w=c+id$, where $a, b, c, d \in \mathbb{R}$ (i.e., are real numbers).  Show that:

   (a) $\overline{z+w} = \overline{z}+\overline{w}$
   
   (b) $\overline{z-w}=\overline{z}-\overline{w}$
   
   (c) $\overline{z\,w}=\overline{z} \,\overline{w}$
   
   (d) $\overline{z\,/\,w} = \overline{z} \, / \, \overline{w}$
   
Let's again test our answers using python:

In [ ]:
# Check Quick Problem (2): symbolically with sympy if installed (a, b, c, d real), otherwise on 1000 random z, w
try:
    import sympy as sm
    a, b, c, d = sm.symbols('a b c d', real=True)
    z, w = a + sm.I*b, c + sm.I*d
    cj = sm.conjugate
    tests = {'2a': (cj(z + w), cj(z) + cj(w)), '2b': (cj(z - w), cj(z) - cj(w)),
             '2c': (cj(z*w), cj(z)*cj(w)),     '2d': (cj(z/w), cj(z)/cj(w))}
    for name, (lhs, rhs) in tests.items():
        print(f'Test {name}: conj(...) - (...) simplifies to', sm.simplify(lhs - rhs))
except ImportError:
    print('(sympy not installed: testing the identities numerically on 1000 random pairs)')
    rng = np.random.default_rng(0)
    z = rng.normal(size=1000) + 1j*rng.normal(size=1000)
    w = rng.normal(size=1000) + 1j*rng.normal(size=1000)
    cj = np.conj
    tests = {'2a': (cj(z + w), cj(z) + cj(w)), '2b': (cj(z - w), cj(z) - cj(w)),
             '2c': (cj(z*w), cj(z)*cj(w)),     '2d': (cj(z/w), cj(z)/cj(w))}
    for name, (lhs, rhs) in tests.items():
        print(f'Test {name}: largest |difference| =', f'{np.max(np.abs(lhs - rhs)):.1e}')

<a name="3.3"></a>
## 3.3 The Complex Plane

<a name="3.3.1"></a>
### 3.3.1 Points and vectors

We are familiar with the real line as a geometric device for visualizing real numbers.  Since the complex number $z=x+iy$ is actually two dimensional, it is not surprising that many consider complex numbers as points on a plane.  For example, the point $z=3+4i$ can be represented on the complex plane as in Figure 3.1.

In [ ]:
C = DSP_COLORS
z = 3 + 4j
r, th = np.abs(z), np.angle(z)
fig, ax = plt.subplots(figsize=(6, 6))
t = np.linspace(0, 2*np.pi, 300)
ax.plot(np.cos(t), np.sin(t), '--', color=C['unit_circle'], lw=1.2, label='unit circle $|z|=1$')
ax.axhline(0, color='k', lw=0.8); ax.axvline(0, color='k', lw=0.8)
ax.annotate('', xy=(z.real, z.imag), xytext=(0, 0),
            arrowprops=dict(arrowstyle='-|>', color=C['time_domain'], lw=2.2, mutation_scale=18))
ax.plot([z.real, z.real], [0, z.imag], ':', color=C['neutral'], lw=1.5)
ax.plot([0, z.real], [z.imag, z.imag], ':', color=C['neutral'], lw=1.5)
ax.plot(z.real, z.imag, 'o', color=C['freq_domain'], ms=9, zorder=5)
ta = np.linspace(0, th, 60)
ax.plot(1.4*np.cos(ta), 1.4*np.sin(ta), color=C['highlight'], lw=2)
ax.text(1.62*np.cos(th/2), 1.62*np.sin(th/2), f'$\\theta$ = {th:.3f} rad\n({np.degrees(th):.2f}°)',
        color=C['highlight'], fontsize=11, va='center')
ax.text(z.real/2 - 1.45, z.imag/2 + 0.3, f'$r=|z|={r:.0f}$', color=C['time_domain'], fontsize=13)
ax.text(z.real + 0.15, z.imag + 0.2, '$z=3+4i$', fontsize=14, color=C['freq_domain'])
ax.text(z.real + 0.08, -0.45, '$x=3$', fontsize=12); ax.text(-1.15, z.imag - 0.1, '$y=4$', fontsize=12)
ax.set_aspect('equal'); ax.set_xlim(-1.3, 5.6); ax.set_ylim(-1.3, 5.6)
ax.set_xlabel('Re$(z)$'); ax.set_ylabel('Im$(z)$'); ax.legend(loc='lower right', fontsize=10)
plt.tight_layout(); plt.show()

<b>Figure 3.1.</b> The complex number $z=3+4i$ as a point (red) and as a vector from the origin (blue). The dotted lines are its Cartesian components $x=3$ and $y=4$; the orange arc is its argument $\theta=0.927$ rad, and the dashed curve is the unit circle $|z|=1$, included for reference. <i>(Python-generated.)</i>

<a name="3.3.2"></a>
### 3.3.2 Polar representation

This point can be thought as having Cartesian coordinates $(x, y) = (3, 4)$ or as the vector drawn from the origin to this point.  The complex number $z=3+4i$ can also be defined by **polar coordinate** variables, $r$ and $\theta$, which represent the vector magnitude from the origin of the complex plane and the angle made with respect to the $x$-axis, respectively. Recall that for $z=x+iy$ we have the following forward and inverse coordinate transformations

**Forward (Cartesian to Polar)**

$$r = \sqrt{x^2+y^2} \tag{14}$$

$$\theta = \arg(z) = {\rm atan2}(y, x) \tag{15}$$

<div class="alert alert-warning">
<b>Common misconception:</b> "The argument is $\mathrm{atan}(y/x)$." The ratio $y/x$ forgets which quadrant $z$ is in. For $z=-3-4i$, $\mathrm{atan}(4/3)=0.927$ rad (first quadrant), but the true argument is $\mathrm{atan2}(-4,-3)=-2.214$ rad (third quadrant). Use the two-argument <code>np.arctan2(y, x)</code>, or <code>np.angle(z)</code> for a complex array.
</div>

**Inverse (Polar to Cartesian)**

$$x = r \, \cos\theta \tag{16}$$

$$y = r \, \sin\theta \tag{17}$$

Thus, for the example $z=3+4i$ we have $r=\sqrt{3^2+4^2}=5$ and $\theta=\mathrm{atan2}(4, 3) = 0.93$ radians.  As in analytic geometry $\theta$ is not unique and we can add to $\theta$ any multiple of $2\pi$ radians:

$$\theta = 0.93+2\pi n,\quad n=0,\pm 1,\pm 2, ... \tag{18}$$

We often call $r$ the **modulus** of $z$ and denote it by $|z|$.  The angle $\theta$ is called the **argument** of $z$ and is denoted $\arg(z)=\theta$ or sometimes just $\angle(z)=\theta$.  It would come as no surprise that complex numbers can be represented by vectors, and that the geometric description of addition and subtraction of vectors is similar to that you would have experienced in other courses.  

To help you visualize the unit circle, Figure 3.2 marks the key angles.

In [ ]:
from fractions import Fraction
C = DSP_COLORS
_tab = {0.0: '0', 0.5: r'\frac{1}{2}', round(np.sqrt(2)/2, 6): r'\frac{\sqrt{2}}{2}',
        round(np.sqrt(3)/2, 6): r'\frac{\sqrt{3}}{2}', 1.0: '1'}
def _exact(v):
    k = round(abs(v), 6); k = 0.0 if k < 1e-9 else k
    return ('-' if v < -1e-9 else '') + _tab[k]
def _radlab(deg):
    fr = Fraction(deg, 180)
    if fr == 0:
        return '0'
    num = r'\pi' if fr.numerator == 1 else f'{fr.numerator}\\pi'
    return num if fr.denominator == 1 else rf'\frac{{{num}}}{{{fr.denominator}}}'
angles = sorted(set(range(0, 360, 30)) | set(range(45, 360, 90)))
fig, ax = plt.subplots(figsize=(8.4, 8))
t = np.linspace(0, 2*np.pi, 400)
ax.plot(np.cos(t), np.sin(t), color=C['unit_circle'], lw=1.5)
ax.axhline(0, color='k', lw=0.8); ax.axvline(0, color='k', lw=0.8)
for d in angles:
    a = np.radians(d); x, y = np.cos(a), np.sin(a)
    fam45 = (d % 90 == 45)
    col = C['freq_domain'] if fam45 else (C['time_domain'] if d % 90 else 'k')
    ax.plot([0, x], [0, y], color=col, lw=0.6, alpha=0.5)
    ax.plot(x, y, 'o', color=col, ms=7, zorder=5)
    rin = 0.50 if fam45 else 0.80
    ax.text(rin*x, rin*y, f'${_radlab(d)}$\n{d}°', ha='center', va='center', fontsize=9, color=col,
            bbox=dict(boxstyle='round,pad=0.1', fc='white', ec='none', alpha=0.85))
    ax.text(1.34*x, 1.22*y, f'$({_exact(x)},\\,{_exact(y)})$', ha='center', va='center', fontsize=10, color=col)
ax.set_aspect('equal'); ax.set_xlim(-1.66, 1.66); ax.set_ylim(-1.42, 1.42)
ax.set_xlabel(r'Re$(z)=\cos\theta$'); ax.set_ylabel(r'Im$(z)=\sin\theta$'); ax.grid(False)
plt.tight_layout(); plt.show()

<b>Figure 3.2.</b> The unit circle with the 16 special angles. Inside: the angle in radians and degrees; outside: the exact point $(\cos\theta, \sin\theta)$. Multiples of $\pi/6$ are blue, multiples of $\pi/4$ red, and the axes black. These values are worth memorizing for this course. <i>(Python-generated.)</i>

<b>Quick Problems</b>
(3) Express the following numbers in the form $re^{i\theta}$ with $r>0$ and $0\le\theta<2\pi$.

* $5i$
* $-7+7i$
* $-4-4\sqrt{3}i$
* $-1$

(4) Express the following numbers in the form $z=x+iy$

* $2e^{i3\pi}$
* $e^{i13\pi/4}$
* $7e^{-i\pi/6}$
* $e^{-i\pi/2}$

<a name="3.3.3"></a>
### 3.3.3 Regions in the complex plane

In this course we will sometimes discuss **regions within the complex plane**.  For example: 
* (1) the open disk of radius 1 centered at $c=1+2i$, $|z-c|<1$
* (2) the complement of the disk in (1)
* (3) the right half plane 
* (4) the upper half plane

Figure 3.3 shades these four regions. A dashed boundary means the boundary points are <i>not</i> part of the region (an open set; see below).

In [ ]:
C = DSP_COLORS
x = np.linspace(-5, 5, 2001); y = np.linspace(-3.5, 3.5, 1401)
X, Y = np.meshgrid(x, y); Z = X + 1j*Y
c = 1 + 2j
regions = [(np.abs(Z - c) < 1, '(a) open disk $|z-c|<1$, $c=1+2i$'),
           (np.abs(Z - c) > 1, '(b) complement $|z-c|>1$'),
           (Z.real > 0,        '(c) right half-plane Re$(z)>0$'),
           (Z.imag > 0,        '(d) upper half-plane Im$(z)>0$')]
fig, axs = plt.subplots(2, 2, figsize=(12, 8.6))
t = np.linspace(0, 2*np.pi, 400)
for ax, (mask, title) in zip(axs.flat, regions):
    ax.contourf(X, Y, mask.astype(float), levels=[0.5, 1.5], colors=[C['passband']])
    ax.axhline(0, color='k', lw=0.8); ax.axvline(0, color='k', lw=0.8)
    ax.set_aspect('equal'); ax.set_xlim(-5, 5); ax.set_ylim(-3.5, 3.5)
    ax.set_title(title, fontsize=12); ax.set_xlabel('Re$(z)$'); ax.set_ylabel('Im$(z)$')
for ax in axs.flat[:2]:   # open sets: boundary circle drawn dashed (excluded)
    ax.plot(c.real + np.cos(t), c.imag + np.sin(t), '--', color=C['time_domain'], lw=1.8)
axs[1, 0].axvline(0, ls='--', color=C['time_domain'], lw=1.8)
axs[1, 1].axhline(0, ls='--', color=C['time_domain'], lw=1.8)
plt.tight_layout(); plt.show()

<b>Figure 3.3.</b> Four regions of the complex plane (shaded): (a) the open disk $|z-c|<1$ with $c=1+2i$; (b) its complement $|z-c|>1$; (c) the right half-plane; (d) the upper half-plane. All four are open sets, so their boundaries (dashed) are excluded. <i>(Python-generated.)</i>

Note that there is a difference between an **open set** and a **closed set**.  An open set is a 2D region with the boundary points removed.  For example, the inequality $|z|<r$ would be an open set while $|z|\le r$ would be a closed set.

<b>Quick Problems</b>
(5) What would the following regions look like:

   * An annulus defined by $r < | z-c | < R$


   * An infinite open sector defined by $\alpha < \arg(z) < \beta$


   * A finite open sector defined by $\alpha < \arg(z) < \beta$ and $r<|z|<R$


(6) Describe the following regions:

   * $|3z-6|\le 9$


   * $\left|\frac{z-1}{1+z}\right|<1$


   * $|z| \le |1-z|$

<a name="3.4"></a>
## 3.4 The Complex Exponential

In previous courses you would have seen that functions of $e^x$, $\mathrm{sin} \, x $ and $\mathrm{cos} \,x $ can be expanded in power series:

$$e^x = 1 + \frac{x}{1!} + \frac{x^2}{2!}+ \frac{x^3}{3!} + \frac{x^4}{4!}+ \mathcal{O}\left(x^5\right) \tag{19}$$

$$\mathrm{sin} \, x = x - \frac{x^3}{3!} + \frac{x^5}{5!} - \frac{x^7}{7!} + \frac{x^9}{9!} + \mathcal{O}\left(x^{11}\right)\tag{20}$$

$$\mathrm{cos} \,x = 1 - \frac{x^2}{2!} + \frac{x^4}{4!}- \frac{x^6}{6!}+ \frac{x^8}{8!} + \mathcal{O}\left(x^{10}\right) \tag{21}$$

where, e.g., $\mathcal{O}(x^5)$ means "terms of order $x^5$ and greater", and $!$ is the factorial operator. If we now set $x=i\theta$ and substitute this into $e^x$ we get:

$$
\begin{eqnarray}
e^{i\theta} &=& 1 + \frac{{i\theta}}{1!} + \frac{({i\theta})^2}{2!}+ \frac{({i\theta})^3}{3!}+ \frac{({i\theta})^4}{4!}+ \frac{({i\theta})^5}{5!}+ \mathcal{O}\left(\theta^{6}\right) \\
\, &=&1 + \frac{{i\theta}}{1!} - \frac{{\theta}^2}{2!}- \frac{i{\theta}^3}{3!}+ \frac{{\theta}^4}{4!}+ \frac{i{\theta}^5}{5!} + \mathcal{O}\left(\theta^{6}\right)\\
\, &=&\left(  1 - \frac{{\theta}^2}{2!} + \frac{{\theta}^4}{4!} +\mathcal{O}\left(\theta^{6}\right)\right) + i\left(\frac{{\theta}}{1!} - \frac{{\theta}^3}{3!} + \frac{{\theta}^5}{5!} + \mathcal{O}\left(\theta^{7}\right) \right) \tag{22}\\
\, &=& \mathrm{cos}\,\theta + i \mathrm{sin}\,\theta \\
\end{eqnarray}
$$

This is **Euler's formula** that defines $e^{i\theta}$ for any given real-valued $\theta$ as a vector of unit length making angle $\theta$ with the real axis.  Alternatively, the Euler Formulae for the cos $\theta$ and sin $\theta$ functions in terms of complex exponentials are given by:

$$
{\rm cos}\,\theta = \frac{{\rm e}^{i\theta} + {\rm e}^{-i\theta}}{2} \quad \mathrm{and} \quad {\rm sin}\,\theta = \frac{{\rm e}^{i\theta} - {\rm e}^{-i\theta}}{2i} \tag{23}
$$

<div class="alert alert-info">
<b>Synthesis &mdash; complex exponentials as eigenfunctions.</b> In MATH 225 you solved $y'' + \omega_0^2 y = 0$ by trying $y = e^{rt}$,
found $r = \pm i\omega_0$, and combined $e^{i\omega_0 t}$ and $e^{-i\omega_0 t}$ into $\cos\omega_0 t$ and $\sin\omega_0 t$: equation (23) read
backwards. The trial works because differentiating $e^{rt}$ returns the same function multiplied by a number, $r$. A function that an operation only
rescales is called an <i>eigenfunction</i> of that operation, and this property of $e^{i\omega t}$ is the reason it appears in every transform in this course.
</div>

Note that multiplying $e^{i\theta}$ by $r$ gives $re^{i\theta}$ which is a complex number having modulus $r$ and argument $\theta$.   Thus, $z=re^{i\theta}$ is convenient when we wish to display the **polar coordinates** of $z$ and the notation $z=x+iy$ is convenient when we want to display the **Cartesian components** of $z$.  For example, all of the following are equivalent representations:

$$1+i = \sqrt{2}e^{i\pi/4} = -\sqrt{2} e^{i5\pi/4} = -\sqrt{2} e^{-i3\pi/4} \tag{24}$$

The series in equation (22) converges quickly. Figure 3.4 adds its terms one at a time for $\theta = 2$ rad: each term is the
previous one turned by $90^\circ$ and shrunk, so the path spirals onto the point $e^{2i}$ on the unit circle.

In [ ]:
C = DSP_COLORS
from math import factorial
theta = 2.0
terms = np.array([(1j*theta)**k / factorial(k) for k in range(21)])
S = np.concatenate([[0], np.cumsum(terms)])
target = np.exp(1j*theta)
fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw=dict(width_ratios=[1.15, 1]))
t = np.linspace(0, 2*np.pi, 300)
axa.plot(np.cos(t), np.sin(t), '--', color=C['unit_circle'], lw=1)
axa.axhline(0, color='k', lw=0.8); axa.axvline(0, color='k', lw=0.8)
for k in range(12):
    axa.annotate('', xy=(S[k+1].real, S[k+1].imag), xytext=(S[k].real, S[k].imag),
                 arrowprops=dict(arrowstyle='-|>', color=plt.cm.viridis(k/11), lw=1.8, mutation_scale=12))
for k, (dx, dy) in enumerate([(-0.1, -0.32), (0.1, 0.0), (-0.1, -0.38), (-0.5, 0.0)]):
    axa.text((S[k].real + S[k+1].real)/2 + dx, (S[k].imag + S[k+1].imag)/2 + dy,
             f'$\\frac{{(i\\theta)^{k}}}{{{k}!}}$', fontsize=12)
axa.plot(target.real, target.imag, 'o', color=C['freq_domain'], ms=9, zorder=5)
axa.text(target.real - 0.85, target.imag + 0.2, r'$e^{2i}$', color=C['freq_domain'], fontsize=14)
axa.set_aspect('equal'); axa.set_xlim(-2.1, 1.25); axa.set_ylim(-1.15, 2.2)
axa.set_xlabel('Re'); axa.set_ylabel('Im'); axa.set_title(r'(a) Partial sums of $e^{i\theta}$, $\theta=2$ rad')
N = np.arange(21)
axb.semilogy(N, np.abs(S[1:] - target), 'o-', color=C['time_domain'], ms=5)
axb.set_xlabel('number of terms $N$'); axb.set_ylabel(r'$|S_N - e^{2i}|$')
axb.set_title('(b) Error of the partial sum'); axb.set_xlim(-0.5, 20.5); axb.set_ylim(1e-16, 5)
plt.tight_layout(); plt.show()

<b>Figure 3.4.</b> Euler's formula as a sum. (a) The terms $(i\theta)^k/k!$ of the series for $e^{i\theta}$, $\theta=2$ rad, drawn head to tail (colors run from the first term to the twelfth); the path spirals onto $e^{2i}$ (red) on the unit circle. (b) The distance between the partial sum $S_N$ and $e^{2i}$: 0.086 after 5 terms, 5.1e-05 after 10, and machine precision by 20. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Note:</b> Equations (19)&ndash;(21) are the MATH 213 Taylor series, which converge for every real $x$; the same series converge for
complex arguments, which is what allows the substitution $x = i\theta$ in equation (22).
</div>

<div class="alert alert-info">
<b>Note:</b> The tip of $e^{i\omega t}$ moves in uniform circular motion with angular speed $\omega$, exactly as in PHGN 100. The projections of
uniform circular motion onto two perpendicular axes are simple harmonic motions, $\cos\omega t$ and $\sin\omega t$. A sinusoid is a rotation seen edge-on.
</div>

The animation in Figure 3.5 runs one full period of $e^{i\omega t}$ with $f = 1$ Hz. Watch three things. The tip of the black phasor moves around the
unit circle at a constant rate, completing one turn per second. Its shadow on the real axis (blue) goes back and forth as $\cos\omega t$, and its shadow on
the imaginary axis (red) as $\sin\omega t$, a quarter period later. And the dashed gray phasor $e^{-i\omega t}$ turns the other way: the imaginary parts of
the two phasors always cancel, and their average is the blue shadow, $\cos\omega t$. That is equation (23) in motion.

In [ ]:
C = DSP_COLORS
n_frames, f = 30, 1.0                      # one full turn of a 1 Hz phasor, 12 degrees per frame
w = 2*np.pi*f
tk = np.arange(n_frames) / n_frames        # 0 ... 29/30 s
tt = np.linspace(0, 1, 301)
th = np.linspace(0, 2*np.pi, 300)
fig = plt.figure(figsize=(10, 4.8))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.6])
axa = fig.add_subplot(gs[0]); axb = fig.add_subplot(gs[1])
axa.plot(np.cos(th), np.sin(th), color=C['unit_circle'], lw=1)
axa.axhline(0, color='k', lw=0.6); axa.axvline(0, color='k', lw=0.6)
axa.set_aspect('equal'); axa.set_xlim(-1.2, 1.2); axa.set_ylim(-1.2, 1.2)
axa.set_xlabel('Re'); axa.set_ylabel('Im')
axb.plot(tt, np.cos(w*tt), color=C['time_domain'], alpha=0.15)
axb.plot(tt, np.sin(w*tt), color=C['freq_domain'], alpha=0.15)
axb.set_xlim(0, 1); axb.set_ylim(-1.2, 1.2); axb.set_xlabel('t [s]'); axb.set_ylabel('amplitude')
ph,  = axa.plot([], [], color='k', lw=2.5)
cj,  = axa.plot([], [], color=C['neutral'], lw=1.5, ls='--')
pre, = axa.plot([], [], color=C['time_domain'], lw=5, solid_capstyle='butt')
pim, = axa.plot([], [], color=C['freq_domain'], lw=5, solid_capstyle='butt')
lc,  = axb.plot([], [], color=C['time_domain'], label=r'Re $e^{i\omega t}=\cos\omega t$')
ls_, = axb.plot([], [], color=C['freq_domain'], label=r'Im $e^{i\omega t}=\sin\omega t$')
dc,  = axb.plot([], [], 'o', color=C['time_domain'])
ds,  = axb.plot([], [], 'o', color=C['freq_domain'])
axb.legend(loc='upper right', fontsize=9)
title = fig.suptitle('', fontsize=13)

def update(k):
    t = tk[k]; z = np.exp(1j*w*t)
    ph.set_data([0, z.real], [0, z.imag]); cj.set_data([0, z.real], [0, -z.imag])
    pre.set_data([0, z.real], [0, 0]);     pim.set_data([0, 0], [0, z.imag])
    m = tt <= t + 1e-12
    lc.set_data(tt[m], np.cos(w*tt[m])); ls_.set_data(tt[m], np.sin(w*tt[m]))
    dc.set_data([t], [z.real]); ds.set_data([t], [z.imag])
    title.set_text(f'$e^{{i\\omega t}}$ with f = 1 Hz:  t = {t:.3f} s,  $\\theta=\\omega t$ = {np.degrees(w*t):.0f}°')
    return []

update(0)
plt.tight_layout()
ani = animation.FuncAnimation(fig, update, frames=n_frames, interval=120, blit=False, repeat=True)
plt.close(fig)
HTML('<div style="display:flex; justify-content:center;">' + ani.to_jshtml() + '</div>')

<b>Figure 3.5.</b> The rotating phasor $e^{i\omega t}$, $f=1$ Hz, 30 frames over one period. (a) Complex plane: the phasor (black), its real projection (blue) and imaginary projection (red), and the counter-rotating conjugate $e^{-i\omega t}$ (dashed gray). (b) The real and imaginary parts traced against time up to the current instant: $\cos\omega t$ (blue) and $\sin\omega t$ (red). <i>(Python-generated.)</i>

<a name="3.5"></a>
## 3.5 Geometry of Multiplication and Division

Based on the rules from the study of exponents involving real numbers, it is fairly straightforward to suppose that the following operations involving imaginary exponents will hold: $e^{ia}e^{ib}$,  $e^{ia}/e^{ib}$ and  $(e^{ia})^b$ where $a, b \in \mathbb{R}$.

To illustrate this, let's define $z=re^{i\theta}$ and $w=se^{i\gamma}$ where $r, s, \theta, \gamma \in \mathbb{R}$. We can then specify the following multiplication and division operations:

**Multiplication**:  We obtain the following:

$$  z\,w = r\, e^{i\theta} s\,e^{i\gamma} = r\,s\, e^{i\theta}\,e^{i\gamma} = r\,s\, e^{i(\theta+\gamma)} \tag{25}$$

The last argument tells us that the modulus is $|z\,w|=|z|\,|w|$ and the argument is $\arg(zw) = \arg(z)+\arg(w)$.  

**Division:** We obtain the following:

$$\frac{z}{w} = \frac{r\, e^{i\theta}}{s\,e^{i\gamma}} = \frac{r}{s}\, e^{i(\theta-\gamma)} \tag{26}$$

where the modulus is given by $|z\,/\,w|= |z|\,/\,|w|$ and the argument by $\arg(z\,/\,w)=\arg(z)-\arg(w)$.

In [ ]:
C = DSP_COLORS
z, w = 1 + 1j, 2*np.exp(1j*np.pi/3)
cases = [(z*w, '$zw$', '(a) Multiplication: moduli multiply, angles add'),
         (z/w, '$z/w$', '(b) Division: moduli divide, angles subtract')]
fig, axs = plt.subplots(1, 2, figsize=(12, 5.2))
def arrow(ax, v, col, lab, dx=0.08, dy=0.08):
    ax.annotate('', xy=(v.real, v.imag), xytext=(0, 0),
                arrowprops=dict(arrowstyle='-|>', color=col, lw=2.2, mutation_scale=16))
    ax.text(v.real + dx, v.imag + dy, lab, color=col, fontsize=13)
def arc(ax, a0, a1, rad, col, lab):
    ta = np.linspace(a0, a1, 50); ax.plot(rad*np.cos(ta), rad*np.sin(ta), color=col, lw=1.6)
    am = (a0 + a1)/2
    if am < 0:   # below the real axis: put the label under the arc's end, clear of the x axis and the vector
        ax.text(rad*np.cos(a1) - 0.05, rad*np.sin(a1) - 0.10, lab, color=col, fontsize=11, va='top', ha='center')
    else:
        ax.text((rad + 0.12)*np.cos(am), (rad + 0.12)*np.sin(am), lab, color=col, fontsize=11)
for ax, (v, lab, ttl) in zip(axs, cases):
    ax.axhline(0, color='k', lw=0.8); ax.axvline(0, color='k', lw=0.8)
    arrow(ax, z, C['time_domain'], r'$z=\sqrt{2}\,e^{i\pi/4}$')
    arrow(ax, w, C['impulse_resp'], r'$w=2\,e^{i\pi/3}$', dx=-1.0, dy=0.12)
    arrow(ax, v, C['freq_domain'], lab + f' $=${abs(v):.3f}$\\,e^{{i({np.degrees(np.angle(v)):.0f}°)}}$',
          dx=0.1, dy=0.05 if v.imag > 0 else -0.35)
    arc(ax, 0, np.angle(z), 0.45, C['time_domain'], '45°')
    arc(ax, 0, np.angle(w), 0.75, C['impulse_resp'], '60°')
    arc(ax, 0, np.angle(v), 1.05 if abs(v) > 1.2 else 0.6*abs(v), C['freq_domain'], f'{np.degrees(np.angle(v)):.0f}°')   # arc inside the vector tip
    ax.set_aspect('equal'); ax.set_title(ttl, fontsize=12); ax.set_xlabel('Re'); ax.set_ylabel('Im')
axs[0].set_xlim(-2.3, 1.9); axs[0].set_ylim(-0.4, 3.2)
axs[1].set_xlim(-0.6, 2.3); axs[1].set_ylim(-0.6, 2.1)
plt.tight_layout(); plt.show()

<b>Figure 3.6.</b> Multiplication and division as rotate-and-scale, for $z=1+i=\sqrt{2}\,e^{i\pi/4}$ (blue) and $w=2e^{i\pi/3}$ (green). (a) $zw$ (red) has modulus $\sqrt{2}\cdot 2 = 2.828$ and angle $45^\circ + 60^\circ = 105^\circ$. (b) $z/w$ (red) has modulus $\sqrt{2}/2 = 0.707$ and angle $45^\circ - 60^\circ = -15^\circ$. <i>(Python-generated.)</i>

<div class="alert alert-info">
<b>Synthesis &mdash; gain and phase.</b> In PHGN 200 AC circuits, a sinusoidal current is represented by a phasor, and each circuit element multiplies
it by a complex impedance: $|Z|$ scales the amplitude and $\arg Z$ shifts the phase. That is equation (25): moduli multiply and arguments add.
A gain and a phase shift are the two numbers needed to describe what a linear circuit, or any linear filter in this course, does to one sinusoid.
</div>

<b>Quick Problems</b>
(7) Prove **de Moivre's Theorem**: 

$$ \left(\cos\,\theta+i\sin\,\theta\right)^{n}=\cos\,n\theta+i\sin\,n\theta \tag{27}$$

where $n$ is an integer.  (Hint: it's straightforward with Euler's Identity. You can also do this more formally through an inductive proof that does not rely on Euler's Identity.)

(8) By squaring $e^{i\theta}=\mathrm{cos}\, \theta + i\,\mathrm{sin} \theta$, find formulas for $\mathrm{cos}\,2\theta$ and $\mathrm{sin}\,2\theta$ in terms of $\mathrm{cos}\,\theta$ and $\mathrm{sin}\,\theta$.

(9) Find formulas for $\mathrm{cos}\,3\theta$ and $\mathrm{sin}\,3\theta$ in terms of $\mathrm{cos}\,\theta$ and $\mathrm{sin}\,\theta$.

<a name="3.6"></a>
## 3.6 Roots of a Complex Number

Suppose we are asked for the cube root of $8i$.  We seek $z$ such that $z^3=8i$. First, we recognize that the **principal value** of $i=e^{i\pi/2}$. Recognizing that we can have $2\pi n$ multiples of this (where $n$ is integer) allows us to write:

$$8i = 8 e^{i(\pi/2+2\pi n)},  \quad n=0,\pm 1,\pm 2, ... \tag{28}$$

Thus, we can enforce the cube root on both sides:

$$\begin{eqnarray}
z^3 &=& 8 e^{i(\pi/2+2\pi n)}\tag{29} \\
z   &=& \left(8 e^{i(\pi/2+2\pi n)} \right)^\frac{1}{3} \tag{30}\\
z   &=& 8^\frac{1}{3} \left(e^{i(\pi/2+2\pi n)} \right)^\frac{1}{3} \tag{31}\\
z   &=& 2 e^{i(\pi/6+2\pi n/3)}, \quad n=0,\pm 1,\pm 2, ... \tag{32}\\
\end{eqnarray}$$

It looks like there are an infinite number of different values!  However, let's look at this in complex plane using an [Argand Diagram](http://mathworld.wolfram.com/ArgandDiagram.html).  In Figure 3.7 I have plotted the values of equation (32) for every integer $n$ from $-50$ to $49$.

In [ ]:
C = DSP_COLORS
nn = np.arange(-50, 50)                        # n = -50 ... 49, as in equation for z
roots_all = 2*np.exp(1j*np.pi*(1/6 + 2*nn/3))
roots = 2*np.exp(1j*np.pi*(1/6 + 2*np.arange(3)/3))
fig, ax = plt.subplots(figsize=(6, 6))
t = np.linspace(0, 2*np.pi, 300)
ax.plot(2*np.cos(t), 2*np.sin(t), '--', color=C['unit_circle'], lw=1, label='$|z|=2$')
ax.axhline(0, color='k', lw=0.8); ax.axvline(0, color='k', lw=0.8)
ax.plot(roots_all.real, roots_all.imag, 'o', color=C['neutral'], ms=14, alpha=0.25, label='n = −50 … 49 (100 values)')
for k, r in enumerate(roots):
    ax.annotate('', xy=(r.real, r.imag), xytext=(0, 0),
                arrowprops=dict(arrowstyle='-|>', color=C['freq_domain'], lw=2, mutation_scale=14))
    ax.plot(r.real, r.imag, 'o', color=C['freq_domain'], ms=7)
lab = [r'$n=0$: $\pi/6$', r'$n=1$: $5\pi/6$', r'$n=2$: $3\pi/2$']
off = [(0.1, 0.18), (-1.25, 0.18), (0.15, -0.05)]
for r, l, (dx, dy) in zip(roots, lab, off):
    ax.text(r.real + dx, r.imag + dy, l, fontsize=11, color=C['freq_domain'])
ax.set_aspect('equal'); ax.set_xlim(-2.6, 2.6); ax.set_ylim(-2.6, 2.6)
ax.set_xlabel('Re'); ax.set_ylabel('Im'); ax.legend(loc='upper left', fontsize=9)
plt.tight_layout(); plt.show()

<b>Figure 3.7.</b> The cube roots of $8i$. The 100 values of $z$ for $n=-50,\dots,49$ (gray) fall on only three points of the circle $|z|=2$ (red): $2e^{i\pi/6}=\sqrt{3}+i$, $2e^{i5\pi/6}=-\sqrt{3}+i$ and $2e^{i3\pi/2}=-2i$. <i>(Python-generated.)</i>

In fact, as illustrated in Figure 3.7, there are only three distinct answers!  This should not be surprising since $z^3=8i$ is a [cubic equation](https://en.wikipedia.org/wiki/Cubic_function) and therefore we should expect exactly three solutions.

<b>Quick Problems</b>
(10) Solve the following equations and show the answers on an Argand diagram:

   * $z^2+4=0$

   * $z^3-8=0$

   * $z^4=81(1+i)/\sqrt{2}$

   * $z^3+i=0$

   * $z^6=-64$

<a name="3.7"></a>
## 3.7 Complex Functions

Complex analysis is concerned with complex functions that are differentiable in some defined domain.  However, we should define what we mean by **complex function**.  A function $f$ on a complex domain $S$ assigns to every $z \in S$ a complex number $w$ called the value of $f$ at $z$:

$$ w = f(z) \tag{33}$$

For example, $w = f(z) = z^2+3z$ is a complex function defined for all of $z$, such that the domain $S$ is the whole complex plane.  Generally speaking, $w$ is itself a complex number, $w = u+iv$, where both $u$ and $v$ can be dependent on $z=x+iy$.  Thus, we write:

$$w = f(z) = u(x,y) + iv(x,y) \tag{34}$$

where $u(x,y)$ and $v(x,y)$ are functions of two real variables.

**EXAMPLE**

**Q1:** If $w=f(z)=z^2$, what are the associated $u(x,y)$ and $v(x,y)$?

**A1:** Because $z=x+iy$ we may write the following:

$$w = z^2 = (x+iy)^2 = x^2+2ixy+(iy)^2 = x^2-y^2 + 2ixy$$

Therefore, by inspection we see that $u(x,y) = x^2-y^2$ and $v(x,y) = 2xy$.

<a name="3.7.1"></a>
### 3.7.1 Limits and continuity

A function $f(z)$ is said to have the **limit** $l$ as $z$ approaches some $z_0$:

$$ \lim_{z\rightarrow z_0} f(z) = l \tag{35}$$

if and only if $f$ is defined in a neighborhood of $z_0$ and if the values of $f$ are "close" to $l$ for all $z$ "close" to $z_0$. More precisely, in the disk $|z-z_0|<\delta$ we have $|f(z)-l|<\epsilon$.  This is similar to how limits are defined in the calculus of 1D real variables; however, one big difference is that $z$ may approach $z_0$ **from any direction** (i.e., in $\theta$, not just from the left (-) and right (+) along the real axis).  

A function $f(z)$ is said to be **continuous** at $z=z_0$ if and only if $f(z_0)$ is defined and 

$$\lim_{z\rightarrow z_0} f(z) = f(z_0) \tag{36}$$

$f(z)$ is said to be **continuous in a domain** if it is continuous at each point in the domain.

**EXAMPLE**

**Q1:** Is $f(z)=\sum_{n=1}^{10} z^n$ continuous at $z=0$?

**A1:** Let's look at the following limit:

$$\lim_{z\rightarrow 0}\sum_{n=1}^{10} z^n  = \lim_{r\rightarrow 0}\sum_{n=1}^{10} (re^{i\theta })^n = \lim_{r\rightarrow 0}\sum_{n=1}^{10} r^ne^{in\theta }=0$$

Thus, because the limit approaches $l=0$, the function is continuous at $z=0$.

<b>Quick Problems</b>
(11) Find out (and give a reason) whether $f(z)$ is continuous at $z=0$ if $f(0)=0$ and for $z\neq0$ the function is equal to:
   * $\frac{\Im{(z)}}{(1-|z|)}$ 
   
   
   * $\frac{\Im{(z^2)}}{|z|^2}$
   
(Hint: Use the polar notation, assert $r\rightarrow 0$ and see if this is valid for all $\theta$).

<a name="3.7.2"></a>
### 3.7.2 The derivative of a complex function

**Provided that the limit exists**, the **derivative** of a complex function $f$ at a point $z_0$ is written $f^\prime(z_0)$ and is defined by

$$f^\prime(z_0) = \lim_{\Delta z\rightarrow 0} \frac{f(z_0+\Delta z)-f(z_0)}{\Delta z} \tag{37} $$

Then $f(z)$ is said to be **differentiable** at $z_0$.  If we write $\Delta z = z-z_0$ then we can change the limit argument and denominator as follows:

$$f^\prime(z_0) = \lim_{z\rightarrow z_0} \frac{f(z)-f(z_0)}{z-z_0} \tag{38}$$

Note that differentiability of $f$ at $z_0$ requires that, along whatever path $z$ approaches $z_0$, the quotient approaches the same value.

**EXAMPLES**

**Q1**: Is the function $f(z)=z^2$ differentiable for all $z$? If yes, what is its derivative? 

**A1:** Yes, the function $f(z)=z^2$ is differentiable for all $z$ and has the derivative $f^\prime(z)=2z$ because:

$$ f^\prime(z) = \lim_{\Delta z\rightarrow 0} \frac{ (z+\Delta z)^2 -z^2 }{\Delta z} = \lim_{\Delta z\rightarrow 0} \frac{ z^2+2z\Delta z +\Delta z^2 -z^2 }{\Delta z}= \lim_{\Delta z\rightarrow 0} \left(2z +\Delta z\right) = 2z \tag{39}$$

**Q2**: Is the function $f(z)=\overline{z}$ differentiable for all $z$? If yes, what is its derivative? 

**A2:** No, the function $f(z)=\overline{z}$ is not differentiable because:

$$ \frac{f(z+\Delta z)-f(z)}{\Delta z} = \frac{\overline{z+\Delta z}-\overline{z}}{\Delta z} =\frac{\overline{\Delta z}}{\Delta z} = \frac{\Delta x - i\Delta y}{\Delta x + i \Delta y} \tag{40}$$

If $\Delta y=0$ and we approach $z_0$ along $\Delta x$ then this expression is equal to $1$; however, if $\Delta x=0$ and we approach $z_0$ along $\Delta y$ then the expression is $-1$.  Thus, the limit does not exist and $\overline{z}$ is not differentiable.

Figure 3.8 shows this numerically: it evaluates the difference quotient for a small step $|\Delta z| = 10^{-3}$ taken in every direction.

In [ ]:
C = DSP_COLORS
z0, h = 1 + 1j, 1e-3
phi = np.linspace(0, 2*np.pi, 361)
dz = h*np.exp(1j*phi)
q_sq = ((z0 + dz)**2 - z0**2) / dz
q_cj = (np.conj(z0 + dz) - np.conj(z0)) / dz
fig, (axa, axb) = plt.subplots(1, 2, figsize=(12, 5), layout='constrained')
sc = axa.scatter(q_sq.real, q_sq.imag, c=phi, cmap='twilight', s=12)
axa.plot(2*z0.real, 2*z0.imag, '+', color='k', ms=16, mew=2)
axa.set_title(r'(a) $f(z)=z^2$ at $z_0=1+i$: one limit, $2z_0$', fontsize=12)
axa.set_xlabel('Re'); axa.set_ylabel('Im'); axa.set_aspect('equal')
axa.set_xlim(2 - 3e-3, 2 + 3e-3); axa.set_ylim(2 - 3e-3, 2 + 3e-3)
axa.ticklabel_format(useOffset=False)
axb.scatter(q_cj.real, q_cj.imag, c=phi, cmap='twilight', s=12)
axb.axhline(0, color='k', lw=0.8); axb.axvline(0, color='k', lw=0.8)
axb.set_title(r'(b) $f(z)=\bar z$: the quotient depends on direction', fontsize=12)
axb.set_xlabel('Re'); axb.set_ylabel('Im'); axb.set_aspect('equal'); axb.set_xlim(-1.3, 1.3); axb.set_ylim(-1.3, 1.3)
cb = fig.colorbar(sc, ax=[axa, axb], shrink=0.85); cb.set_label(r'direction $\varphi$ of $\Delta z$ [rad]')
plt.show()

<b>Figure 3.8.</b> Difference quotients $[f(z_0+\Delta z)-f(z_0)]/\Delta z$ for $|\Delta z|=10^{-3}$ in all directions $\varphi$ (color). (a) For $f(z)=z^2$ at $z_0=1+i$ every direction gives (to within $|\Delta z|$) the same value $2z_0=2+2i$ (cross), so $f'(z_0)$ exists. (b) For $f(z)=\bar z$ the quotient is $e^{-2i\varphi}$: it runs around the unit circle, equal to 1 along the real axis and $-1$ along the imaginary axis, so no derivative exists. <i>(Python-generated.)</i>

<b>Quick Problems</b>
(12) Are the following functions differentiable? If so, what is the derivative?:

   * $f(z) = \left(z^3 + i\right)^2 $
   
   
   * $f(z) =\frac{i}{(1-z)^2}$

<a name="3.7.3"></a>
### 3.7.3 Analytic functions

Complex analysis is concerned with the theory and application of [analytic functions](https://en.wikipedia.org/wiki/Analytic_function) - i.e., functions that are differentiable in some domain such that we can "do calculus".  An **analytic function** is said to be analytic in a domain $D$ if $f(z)$ is defined and differentiable at all points in $D$. The function $f(z)$ is said to be **analytic** at a point $z=z_0$ in D if $f(z)$ is analytic in a neighborhood of $z_0$.  

Importantly, non-negative integer powers $\{1, z, z^2, z^3, ...\}$ are analytic in the complex plane and so are [polynomials](https://en.wikipedia.org/wiki/Polynomial) of the form:

$$f(z) = c_0 + c_1z + c_2 z^2 + c_3z^3 + ... + c_nz^n = \sum_{n=0}^{N} c_n z^n \tag{41} $$

The quotient of two polynomials $g(z)$ and $h(z)$:

$$f(z) = \frac{g(z)}{h(z)} \tag{42}$$

is called a [rational function](https://en.wikipedia.org/wiki/Rational_function).  Rational function $f(z)$ is analytic except at points where $h(z)=0$ [also called the poles of $f(z)$].  We also can assume that many common factors of $g$ and $h$ cancel out.  Rational functions will be quite important in the context of designing filters in the later sections of this course (Modules 14 and 15).

<a name="3.7.4"></a>
### 3.7.4 The Cauchy–Riemann equations

Finally, we are going to conclude this overview of complex numbers with a brief discussion of the [Cauchy-Riemann equations](https://en.wikipedia.org/wiki/Cauchy%E2%80%93Riemann_equations), which are arguably the most important equations in [complex analysis](https://en.wikipedia.org/wiki/Complex_analysis).  These equations provide a test criterion for the analyticity of a complex function $w=f(z)= u(x,y)+iv(x,y)$.  Roughly, $f$ is analytic in a domain $D$ if and only if the first partial derivatives of $u$ and $v$ satisfy the two Cauchy-Riemann equations:

$$\begin{eqnarray}
\frac{\partial u}{\partial x} &=& \frac{\partial v}{\partial y} \tag{43}\\
\frac{\partial u}{\partial y} &=&-\frac{\partial v}{\partial x} \tag{44}\\
\end{eqnarray}$$

everywhere in domain $D$.  

**Q1**: Show that $f(z)=z^2$ is analytic for all $z$. 

**A1:** First, we observe that $z^2 =x^2-y^2+i2xy$ and thus $u = x^2-y^2$ and $v=2xy$.  We then apply the Cauchy-Riemann test: 

$$\begin{eqnarray}
\frac{\partial u}{\partial x} = 2x, &\quad \, \quad & \frac{\partial u}{\partial y}=-2y \tag{45}\\
\frac{\partial v}{\partial x} = 2y, &\quad\,\quad & \frac{\partial v}{\partial y}=2x
\tag{46}\\
\end{eqnarray}$$

Thus, $f(z)=z^2$ satisfies the Cauchy-Riemann equations and is analytic for all of $z$.

<div class="alert alert-success">
<h3><center>Distraction A: Analytic functions and potential fields</center></h3>

Differentiate the first Cauchy&ndash;Riemann equation with respect to $x$ and
the second with respect to $y$, and add: $\partial^2 u/\partial x^2 + \partial^2 u/\partial y^2 = 0$. The real (and imaginary) part of any analytic
function satisfies Laplace's equation, the equation the gravitational potential satisfies outside the masses in GPGN 328 (POTE1). For
$f(z)=z^2$, $u = x^2 - y^2$ gives $2 - 2 = 0$. This is why 2D potential-field problems can be solved with analytic functions.
</div>

<b>Quick Problems</b>
(13) Determine whether the following functions are analytic:
   
   * $f(z) = z^4$
   
   
   * $f(z) = e^{2x}\left(\mathrm{cos}\, y + i  \,\mathrm{sin} \, y\right)$

---
# Microquestions

<div class="alert alert-warning">
<b>Microquestion 3.1 &mdash; Two arrivals, one phasor</b>

A geophone records a direct arrival $x_1(t) = 3\cos(2\pi f t)$ and a second arrival of the same frequency $x_2(t) = 4\cos(2\pi f(t - \Delta t))$, delayed by $\Delta t = 0.125$ s, with $f = 2$ Hz. Each sinusoid $A\cos(2\pi f t + \phi)$ is the real part of the phasor $A e^{i\phi}$ times $e^{i2\pi f t}$ (Section 3.5).

<b>(a)</b> Write the phasors of $x_1$ and $x_2$, add them, and give the amplitude and phase of the combined record $x_1 + x_2$.

<b>(b)</b> For a general delay, show that the combined amplitude is $\sqrt{25 + 24\cos(2\pi f\Delta t)}$. Find its largest and smallest values and the smallest delay $\Delta t > 0$ that gives the smallest. Relate this to two-source interference in PHGN 200.

<b>(c)</b> A student reads the two peak amplitudes off the trace, adds them, and reports a combined peak of 7. When is this right, and what is the combined peak for the delay in the setup?
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 3.1</b></summary>

<b>(a)</b> $x_1 \to 3$. $x_2 \to 4e^{-i2\pi f\Delta t} = 4e^{-i\pi/2} = -4i$ (a phase lag of $2\pi \cdot 2 \cdot 0.125 = \pi/2$). Sum: $3 - 4i$, so the amplitude is $\sqrt{9+16} = 5$ and the phase is $\mathrm{atan2}(-4, 3) = -0.927$ rad ($-53.1^\circ$): $x_1 + x_2 = 5\cos(4\pi t - 0.927)$.

<b>(b)</b> $|3 + 4e^{-i\alpha}|^2 = (3 + 4e^{-i\alpha})(3 + 4e^{i\alpha}) = 9 + 16 + 12(e^{i\alpha} + e^{-i\alpha}) = 25 + 24\cos\alpha$, with $\alpha = 2\pi f\Delta t$. Largest $\sqrt{49} = 7$ ($\alpha = 0$, in phase); smallest $\sqrt{1} = 1$ ($\alpha = \pi$, out of phase), first at $\Delta t = 1/(2f) = 0.25$ s. This is constructive and destructive interference from PHGN 200, with the phase difference set by a time delay instead of a path difference.

<b>(c)</b> Adding peak amplitudes is right only when the arrivals are in phase ($\alpha = 0, 2\pi, \dots$). For $\Delta t = 0.125$ s they are a quarter period apart and the combined peak is 5, not 7. Amplitudes add as vectors (phasors, Figure 3.6), not as numbers. Overlapping arrivals on a seismic record (Module 1) behave the same way.
</details>

<div class="alert alert-warning">
<b>Microquestion 3.2 &mdash; A phasor, one sample at a time</b>

A 25 Hz sinusoid is recorded at a sample interval $\Delta t = 5$ ms, so the samples of $e^{i2\pi f t}$ are $e^{i2\pi f n\Delta t} = w^n$ with $w = e^{i2\pi f\Delta t}$ and $n = 0, 1, 2, \dots$ (sampled signals were introduced in Module 2).

<b>(a)</b> Find the angle $\theta$ that the phasor turns between samples, the number of samples per period, and $w^3$ in the form $x + iy$ (use Figure 3.2).

<b>(b)</b> Show that $w^8 = 1$, and use the finite geometric series from MATH 213, $\sum_{n=0}^{N-1} w^n = (1 - w^N)/(1 - w)$, to show that $\sum_{n=0}^{7} w^n = 0$. Interpret this as the average of the sampled phasor over one period.

<b>(c)</b> A classmate claims that $\sum_{n=0}^{7} w^{kn} = 0$ for every integer $k$. Find the values of $k$ for which this fails, give the sum for them, and explain both cases geometrically.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 3.2</b></summary>

<b>(a)</b> $\theta = 2\pi f\Delta t = 2\pi(25)(0.005) = \pi/4$ ($45^\circ$), so there are $2\pi/\theta = 8$ samples per period. $w^3 = e^{i3\pi/4} = -\sqrt{2}/2 + i\sqrt{2}/2 \approx -0.707 + 0.707i$.

<b>(b)</b> $w^8 = e^{i8\pi/4} = e^{i2\pi} = 1$. Then $\sum_{n=0}^{7} w^n = (1 - w^8)/(1 - w) = 0/(1 - w) = 0$, since $w \neq 1$. The eight samples are eight equally spaced points on the unit circle whose vectors cancel: over a whole period the average of $e^{i\omega t}$ is zero.

<b>(c)</b> It fails when $w^k = 1$, i.e., $k$ a multiple of 8 ($k = 0, \pm 8, \pm 16, \dots$). Then every term is 1 and the sum is 8; the geometric-series formula does not apply because its denominator $1 - w^k$ is zero. For any other $k$ the points $w^{kn}$ are still spread symmetrically around the circle (for $k = 2$, four points visited twice; for $k = 4$, the two points $\pm 1$ alternating), so they cancel. These are the roots of unity of Section 3.6.
</details>

<div class="alert alert-warning">
<b>Microquestion 3.3 &mdash; A seismometer's resonance is a pole</b>

A seismometer's mass on a spring obeys the driven, damped oscillator of MATH 225 (and PHGN 100): $\ddot{x} + \gamma\dot{x} + \omega_0^2 x = F e^{i\omega t}$. Trying $x = A e^{i\omega t}$ gives $A(\omega) = F/(\omega_0^2 - \omega^2 + i\gamma\omega)$. Take a natural frequency $f_0 = 1$ Hz ($\omega_0 = 2\pi$ rad/s), damping $\gamma = 2\zeta\omega_0$ with damping ratio $\zeta = 0.1$, and $F = 1$.

<b>(a)</b> Compute $|A|$ and $\arg A$ at $\omega = \omega_0$, and the static response $|A(0)|$. By what factor does the response at resonance exceed the static response?

<b>(b)</b> Replace $i\omega$ by a complex variable $s$, so that $A(s) = 1/(s^2 + \gamma s + \omega_0^2)$ is a rational function (Section 3.7.3). Use the quadratic formula, equation (7), to find its poles. Show that $|A(i\omega_0)| = 1/(|i\omega_0 - s_1|\,|i\omega_0 - s_2|)$ and say which pole controls the size of the peak.

<b>(c)</b> A student argues: "$A(s)$ is a ratio of polynomials, polynomials are analytic everywhere, so $|A(i\omega)|$ can never become very large." Find the flaw, and use (a) and (b) to say what happens to the peak as $\zeta \to 0$.
</div>

<details>
<summary><b>Show answer &mdash; Microquestion 3.3</b></summary>

<b>(a)</b> $\gamma = 2(0.1)(2\pi) = 1.257$ s$^{-1}$. At $\omega = \omega_0$ the real part of the denominator vanishes: $A = 1/(i\gamma\omega_0)$, so $|A| = 1/(1.257 \times 6.283) = 0.1267$ and $\arg A = -90^\circ$. Static: $|A(0)| = 1/\omega_0^2 = 0.02533$. Ratio $0.1267/0.02533 = 5.0 = 1/(2\zeta)$.

<b>(b)</b> $s = [-\gamma \pm \sqrt{\gamma^2 - 4\omega_0^2}]/2 = -\gamma/2 \pm i\omega_0\sqrt{1 - \zeta^2} = -0.628 \pm 6.252i$. Factoring, $s^2 + \gamma s + \omega_0^2 = (s - s_1)(s - s_2)$, so $|A(i\omega_0)| = 1/(|i\omega_0 - s_1|\,|i\omega_0 - s_2|) = 0.1267$, as in (a). The pole $s_1 = -0.628 + 6.252i$ is only 0.629 from the point $i\omega_0$ on the imaginary axis; the other is about 12.6 away and changes little near resonance. The near pole sets the peak.

<b>(c)</b> A rational function is analytic except where its denominator is zero: at the poles (Section 3.7.3). The poles sit a distance $\gamma/2 = \zeta\omega_0$ to the left of the imaginary axis, so as $\zeta \to 0$ they approach it and the resonance peak $1/(2\zeta)$ grows without bound (10 for $\zeta = 0.05$, 50 for $\zeta = 0.01$). This is why the seismometers that record the data in this course (Module 1) are built with strong damping.
</details>